In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import jensenshannon
from scipy.stats import wasserstein_distance

# ============================================================
# CONFIG
# ============================================================

BASE_DIR = Path(r"./../MIMICEL_data")

INPUT_CSV = BASE_DIR / "mimicel.csv"
OUT_DIR = Path(r"./results")

CASE_COL = "stay_id"
ACT_COL = "activity"

# 본 실험 총 case 수
N_TOTAL = 900

# 7:1:2 split
TRAIN_RATIO = 0.7
VAL_RATIO = 0.1
TEST_RATIO = 0.2

SEED_START = 0
SEED_END = 100

# early stopping 기준
# 값이 작을수록 원본 분포와 유사
EARLY_STOP_SCORE = 0.015

# trace length 영향 가중치
LENGTH_WEIGHT = 0.01

os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(INPUT_CSV)

df[CASE_COL] = df[CASE_COL].astype(str)

print("Loaded:", INPUT_CSV)
print("Total events:", len(df))
print("Total cases:", df[CASE_COL].nunique())

# ============================================================
# GLOBAL DISTRIBUTIONS
# ============================================================

all_cases = df[[CASE_COL]].drop_duplicates().reset_index(drop=True)

global_act_dist = (
    df[ACT_COL]
    .value_counts(normalize=True)
    .sort_index()
)

global_trace_lengths = (
    df.groupby(CASE_COL)
    .size()
    .values
)

# ============================================================
# FUNCTIONS
# ============================================================

def get_df_by_cases(data: pd.DataFrame, case_df: pd.DataFrame) -> pd.DataFrame:
    case_set = set(case_df[CASE_COL].astype(str))
    return data[data[CASE_COL].isin(case_set)].copy()


def get_activity_dist(data: pd.DataFrame) -> pd.Series:
    return (
        data[ACT_COL]
        .value_counts(normalize=True)
        .reindex(global_act_dist.index, fill_value=0)
        .sort_index()
    )


def get_trace_lengths(data: pd.DataFrame) -> np.ndarray:
    return data.groupby(CASE_COL).size().values


def evaluate_split(train_df, val_df, test_df):
    train_act = get_activity_dist(train_df)
    val_act = get_activity_dist(val_df)
    test_act = get_activity_dist(test_df)

    train_len = get_trace_lengths(train_df)
    val_len = get_trace_lengths(val_df)
    test_len = get_trace_lengths(test_df)

    act_score = (
        jensenshannon(global_act_dist, train_act)
        + jensenshannon(global_act_dist, val_act)
        + jensenshannon(global_act_dist, test_act)
    )

    len_score = (
        wasserstein_distance(global_trace_lengths, train_len)
        + wasserstein_distance(global_trace_lengths, val_len)
        + wasserstein_distance(global_trace_lengths, test_len)
    )

    total_score = act_score + LENGTH_WEIGHT * len_score

    return total_score, act_score, len_score


# ============================================================
# SEARCH BEST SEED
# ============================================================

n_train = int(N_TOTAL * TRAIN_RATIO)
n_val = int(N_TOTAL * VAL_RATIO)
n_test = N_TOTAL - n_train - n_val

print()
print("Target split")
print("Train cases:", n_train)
print("Val cases:", n_val)
print("Test cases:", n_test)

best = None
history = []

for seed in range(SEED_START, SEED_END + 1):
    sampled_cases = all_cases.sample(
        n=N_TOTAL,
        random_state=seed
    ).reset_index(drop=True)

    train_cases, temp_cases = train_test_split(
        sampled_cases,
        train_size=n_train,
        random_state=seed,
        shuffle=True
    )

    val_cases, test_cases = train_test_split(
        temp_cases,
        train_size=n_val,
        test_size=n_test,
        random_state=seed,
        shuffle=True
    )

    train_df = get_df_by_cases(df, train_cases)
    val_df = get_df_by_cases(df, val_cases)
    test_df = get_df_by_cases(df, test_cases)

    total_score, act_score, len_score = evaluate_split(
        train_df, val_df, test_df
    )

    row = {
        "seed": seed,
        "total_score": total_score,
        "activity_js_score": act_score,
        "trace_length_wasserstein_score": len_score,
        "train_cases": train_df[CASE_COL].nunique(),
        "val_cases": val_df[CASE_COL].nunique(),
        "test_cases": test_df[CASE_COL].nunique(),
        "train_events": len(train_df),
        "val_events": len(val_df),
        "test_events": len(test_df),
    }

    history.append(row)

    print(
        f"[seed={seed:03d}] "
        f"score={total_score:.6f} | "
        f"act={act_score:.6f} | "
        f"len={len_score:.6f}"
    )

    if best is None or total_score < best["total_score"]:
        best = {
            **row,
            "train_cases_df": train_cases,
            "val_cases_df": val_cases,
            "test_cases_df": test_cases,
            "train_df": train_df,
            "val_df": val_df,
            "test_df": test_df,
        }

    if total_score <= EARLY_STOP_SCORE:
        print()
        print("Early stopping triggered.")
        print("Seed:", seed)
        print("Score:", total_score)
        break

# ============================================================
# SAVE RESULTS
# ============================================================

history_df = pd.DataFrame(history)
history_df.to_csv(OUT_DIR / "split_seed_search_history.csv", index=False)

best["train_df"].to_csv(OUT_DIR / "mimicel_train.csv", index=False)
best["val_df"].to_csv(OUT_DIR / "mimicel_val.csv", index=False)
best["test_df"].to_csv(OUT_DIR / "mimicel_test.csv", index=False)

best["train_cases_df"].to_csv(OUT_DIR / "train_stay_ids.csv", index=False)
best["val_cases_df"].to_csv(OUT_DIR / "val_stay_ids.csv", index=False)
best["test_cases_df"].to_csv(OUT_DIR / "test_stay_ids.csv", index=False)

# activity distribution 비교표 저장
dist_summary = pd.DataFrame({
    "global": global_act_dist,
    "train": get_activity_dist(best["train_df"]),
    "val": get_activity_dist(best["val_df"]),
    "test": get_activity_dist(best["test_df"]),
})

dist_summary["train_diff"] = dist_summary["train"] - dist_summary["global"]
dist_summary["val_diff"] = dist_summary["val"] - dist_summary["global"]
dist_summary["test_diff"] = dist_summary["test"] - dist_summary["global"]

dist_summary.to_csv(OUT_DIR / "activity_distribution_summary.csv")

# trace length 요약 저장
length_summary = pd.DataFrame({
    "split": ["global", "train", "val", "test"],
    "n_cases": [
        df[CASE_COL].nunique(),
        best["train_df"][CASE_COL].nunique(),
        best["val_df"][CASE_COL].nunique(),
        best["test_df"][CASE_COL].nunique(),
    ],
    "n_events": [
        len(df),
        len(best["train_df"]),
        len(best["val_df"]),
        len(best["test_df"]),
    ],
    "mean_trace_length": [
        np.mean(global_trace_lengths),
        np.mean(get_trace_lengths(best["train_df"])),
        np.mean(get_trace_lengths(best["val_df"])),
        np.mean(get_trace_lengths(best["test_df"])),
    ],
    "median_trace_length": [
        np.median(global_trace_lengths),
        np.median(get_trace_lengths(best["train_df"])),
        np.median(get_trace_lengths(best["val_df"])),
        np.median(get_trace_lengths(best["test_df"])),
    ],
    "max_trace_length": [
        np.max(global_trace_lengths),
        np.max(get_trace_lengths(best["train_df"])),
        np.max(get_trace_lengths(best["val_df"])),
        np.max(get_trace_lengths(best["test_df"])),
    ],
})

length_summary.to_csv(OUT_DIR / "trace_length_summary.csv", index=False)

# ============================================================
# PRINT FINAL
# ============================================================

print()
print("========== BEST SPLIT ==========")
print("Best seed:", best["seed"])
print("Total score:", best["total_score"])
print("Activity JS score:", best["activity_js_score"])
print("Trace length Wasserstein score:", best["trace_length_wasserstein_score"])

print()
print("Cases")
print("Train:", best["train_df"][CASE_COL].nunique())
print("Val:", best["val_df"][CASE_COL].nunique())
print("Test:", best["test_df"][CASE_COL].nunique())

print()
print("Events")
print("Train:", len(best["train_df"]))
print("Val:", len(best["val_df"]))
print("Test:", len(best["test_df"]))

print()
print("Saved to:")
print(OUT_DIR)

Loaded: ..\MIMICEL_data\mimicel.csv
Total events: 7568824
Total cases: 425028

Target split
Train cases: 630
Val cases: 90
Test cases: 180
[seed=000] score=0.085733 | act=0.053022 | len=3.271094
[seed=001] score=0.080513 | act=0.047126 | len=3.338719
[seed=002] score=0.068711 | act=0.040794 | len=2.791714
[seed=003] score=0.054614 | act=0.030075 | len=2.453931
[seed=004] score=0.098341 | act=0.059319 | len=3.902104
[seed=005] score=0.111915 | act=0.068568 | len=4.334685
[seed=006] score=0.065229 | act=0.039364 | len=2.586513
[seed=007] score=0.095149 | act=0.067705 | len=2.744426
[seed=008] score=0.072466 | act=0.048670 | len=2.379569
[seed=009] score=0.109520 | act=0.069120 | len=4.039960
[seed=010] score=0.086372 | act=0.065760 | len=2.061213
[seed=011] score=0.079448 | act=0.044722 | len=3.472607
[seed=012] score=0.073387 | act=0.046807 | len=2.658019
[seed=013] score=0.067810 | act=0.041975 | len=2.583484
[seed=014] score=0.100970 | act=0.067686 | len=3.328400
[seed=015] score=0.10